In [1]:
pip install pandas numpy sqlalchemy pymysql openpyxl

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 3.3 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.3 MB/s  0:00:00

   ---------- ----------------------------- 1/4 [pymysql]
   ---------- ----------------------------- 1/4 [pymysql]
   -------------------- ------------------- 2/4 [greenlet]
   -------------------- ------------------- 2/4 [greenlet]
   -------------------- ------------------- 2/4 [greenlet]
   -------------------- ------------------- 2/4 [greenlet]
   -------------------- ------------------- 2/4 [greenlet]
   -------------------- ------------------- 2/4 [greenlet]
   -------------------- ------------------- 2/4 [greenlet]
   -------------------- ------------------- 2/4 [greenlet]
   ------------------------------ --------- 3/4 

In [5]:
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [3]:
import pandas as pd
df = pd.read_excel('online_retail_II.xlsx', sheet_name=None)  # loads both sheets as dict
df = pd.concat(df.values(), ignore_index=True)

df.info()
df.isnull().sum()
df.describe()
df['Invoice'].astype(str).str.startswith('C').sum()  # cancelled orders count

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


np.int64(19494)

In [7]:
df['StockCode'].unique()

array([85048, '79323P', '79323W', ..., 23609, 23617, 23843],
      shape=(5305,), dtype=object)

In [ ]:
# checking for negative values in quantity and negative or zero values in price
print("Negative Quantity rows:", (df['Quantity'] < 0).sum())
print("Negative/Zero Price rows:", (df['Price'] <= 0).sum())
print("Duplicate rows:", df.duplicated().sum())

Negative Quantity rows: 22950
Negative/Zero Price rows: 6207
Duplicate rows: 34335


In [8]:
# Work on a copy so raw data stays untouched (good practice, mention this in interview)
df_clean = df.copy()

# 1. Standardize column names
df_clean.rename(columns={'Customer ID': 'CustomerID'}, inplace=True)

# 2. Flag cancellations instead of blindly dropping (shows business thinking)
df_clean['IsCancelled'] = df_clean['Invoice'].astype(str).str.startswith('C')

# 3. Remove rows with missing Description (small number, not usable)
df_clean = df_clean.dropna(subset=['Description'])

# 4. Handle missing CustomerID — flag as guest instead of dropping (preserves revenue data)
df_clean['CustomerID'] = df_clean['CustomerID'].fillna(0).astype('int64')
df_clean['IsGuestOrder'] = df_clean['CustomerID'] == 0

# 5. Remove rows with negative/zero price (data errors, not real transactions)
df_clean = df_clean[df_clean['Price'] > 0]

# 6. Separate returns (negative quantity, non-cancelled) from actual errors
df_clean['IsReturn'] = (df_clean['Quantity'] < 0) & (~df_clean['IsCancelled'])

# 7. Remove exact duplicate rows
df_clean = df_clean.drop_duplicates()

# 8. Create derived columns for analysis
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']
df_clean['OrderMonth'] = df_clean['InvoiceDate'].dt.to_period('M').astype(str)
df_clean['OrderYear'] = df_clean['InvoiceDate'].dt.year
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()

# 9. Remove StockCode entries that are administrative, not products (POST, DOT, M, etc.)
non_product_codes = ['POST', 'DOT', 'M', 'BANK CHARGES', 'CRUK', 'C2', 'PADS']
df_clean = df_clean[~df_clean['StockCode'].isin(non_product_codes)]

print(f"Original rows: {len(df)}")
print(f"Cleaned rows: {len(df_clean)}")
print(f"Rows removed: {len(df) - len(df_clean)}")

Original rows: 1067371
Cleaned rows: 1021732
Rows removed: 45639


In [9]:
df_clean.to_csv('online_retail_cleaned.csv', index=False)
print("Saved successfully")

Saved successfully
